In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.config('spark.driver.host','localhost').appName('02').getOrCreate()

In [3]:
fifa = spark.read.csv('C:\\datas\\3rd_week_data\\fifa19.csv',header=True,inferSchema=True)
fifa.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Photo: string (nullable = true)
 |-- Nationality: string (nullable = true)
 |-- Flag: string (nullable = true)
 |-- Overall: integer (nullable = true)
 |-- Potential: integer (nullable = true)
 |-- Club: string (nullable = true)
 |-- Club Logo: string (nullable = true)
 |-- Value: string (nullable = true)
 |-- Wage: string (nullable = true)
 |-- Special: integer (nullable = true)
 |-- Preferred Foot: string (nullable = true)
 |-- International Reputation: integer (nullable = true)
 |-- Weak Foot: integer (nullable = true)
 |-- Skill Moves: integer (nullable = true)
 |-- Work Rate: string (nullable = true)
 |-- Body Type: string (nullable = true)
 |-- Real Face: string (nullable = true)
 |-- Position: string (nullable = true)
 |-- Jersey Number: integer (nullable = true)
 |-- Joined: string (nullable = true)
 |-- Loaned From: string (nu

In [4]:
fifa.show(3)

+---+------+-----------------+---+--------------------+-----------+--------------------+-------+---------+-------------------+--------------------+-------+-----+-------+--------------+------------------------+---------+-----------+--------------+----------+---------+--------+-------------+------------+-----------+--------------------+------+------+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+--------+---------+---------------+------------+-------+---------+-----+----------+-----------+-----------+------------+-----------+-------+---------+-------+---------+-------+-------+--------+---------+----------+-------------+-----------+------+---------+---------+-------+--------------+-------------+--------+----------+---------+-------------+----------+--------------+
|_c0|    ID|             Name|Age|               Photo|Nationality|                Flag|Overall|Potential|               Club|           Club Log

## `select`와 `orderBy`
- `select`: 특정 column만 추출함
> - 반환 결과의 type은 `DataFrame`
> - 기존 `DataFrame` column의 특성을 그대로 가져옴(data type, nullable)
> - `select`는 `column`명(`'Age','Name'`)을 사용할 수도 있고, `column` 객체(`fifa['Age'] 또는 fifa.Age`)를 지정할 수 도 있음
- `orderBy`: 지정한 column 기준으로 정렬

In [5]:
# Name, Age, Club 만 추출
fifa_1 = fifa.select('Name','Age','Club') # column명을 사용하여 지정
print(type(fifa_1))
fifa_1.printSchema()
fifa_1.show(3)

<class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Club: string (nullable = true)

+-----------------+---+-------------------+
|             Name|Age|               Club|
+-----------------+---+-------------------+
|         L. Messi| 31|       FC Barcelona|
|Cristiano Ronaldo| 33|           Juventus|
|        Neymar Jr| 26|Paris Saint-Germain|
+-----------------+---+-------------------+
only showing top 3 rows



In [6]:
fifa_1 = fifa.select(fifa['Name'],fifa['Age'],fifa['Club']) # column 객체 자체를 지정
fifa_1.printSchema()
fifa_1.show(3)

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Club: string (nullable = true)

+-----------------+---+-------------------+
|             Name|Age|               Club|
+-----------------+---+-------------------+
|         L. Messi| 31|       FC Barcelona|
|Cristiano Ronaldo| 33|           Juventus|
|        Neymar Jr| 26|Paris Saint-Germain|
+-----------------+---+-------------------+
only showing top 3 rows



In [7]:
fifa_1 = fifa.select(fifa.Name,fifa.Age,fifa.Club) # column 객체 자체를 지정
# column명에 공백이 포함된 경우 위 방식으로 column을 지정할 수 없음
# ex> fifa.full name (X) -> fifa['full name'] (O)
fifa_1.printSchema()
fifa_1.show(3)

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Club: string (nullable = true)

+-----------------+---+-------------------+
|             Name|Age|               Club|
+-----------------+---+-------------------+
|         L. Messi| 31|       FC Barcelona|
|Cristiano Ronaldo| 33|           Juventus|
|        Neymar Jr| 26|Paris Saint-Germain|
+-----------------+---+-------------------+
only showing top 3 rows



- `sql.functions.col` 함수를 사용하여, column 객체를 추출할 수도 있음

In [8]:
from pyspark.sql.functions import col
fifa_1 = fifa.select(col('Name'),col('Age'),col('Club')) # col함수를 사용하여 column 객체 자체를 지정
fifa_1.printSchema()
fifa_1.show(3)

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Club: string (nullable = true)

+-----------------+---+-------------------+
|             Name|Age|               Club|
+-----------------+---+-------------------+
|         L. Messi| 31|       FC Barcelona|
|Cristiano Ronaldo| 33|           Juventus|
|        Neymar Jr| 26|Paris Saint-Germain|
+-----------------+---+-------------------+
only showing top 3 rows



In [9]:
# orderBy: 지정된 열의 값을 기준으로 오름차순 정렬
fifa_2 = fifa_1.orderBy('Age')
fifa_2.show(3)

+-----------+---+------------+
|       Name|Age|        Club|
+-----------+---+------------+
|  B. Nygren| 16|IFK Göteborg|
|W. Geubbels| 16|   AS Monaco|
|   B. Mumba| 16|  Sunderland|
+-----------+---+------------+
only showing top 3 rows



In [10]:
# orderBy도 column명과 column객체 모두 사용 가능
fifa_2 = fifa_1.orderBy(fifa_1.Age)
fifa_2.show(3)

+-----------+---+------------+
|       Name|Age|        Club|
+-----------+---+------------+
|  B. Nygren| 16|IFK Göteborg|
|W. Geubbels| 16|   AS Monaco|
|   B. Mumba| 16|  Sunderland|
+-----------+---+------------+
only showing top 3 rows



- **내림차순** 정렬은???
> - `orderBy(column객체.desc())`: `desc()` 사용할 경우, column명으로 지정할수는 없음

In [11]:
# fifa_1.orderBy(fifa_1.Age.desc()).show(10)
fifa_1.orderBy(fifa_1['Age'].desc()).show(10)

+-------------+---+--------------------+
|         Name|Age|                Club|
+-------------+---+--------------------+
|     O. Pérez| 45|             Pachuca|
|    T. Warner| 44|  Accrington Stanley|
|K. Pilkington| 44|    Cambridge United|
|  S. Narazaki| 42|      Nagoya Grampus|
|    J. Villar| 41|                NULL|
| H. Sulaimani| 41|           Ohod Club|
|     B. Nivet| 41|        ESTAC Troyes|
|     M. Tyler| 41| Peterborough United|
|     C. Muñoz| 41|CD Universidad de...|
|  B. Castillo| 40|      Atlético Huila|
+-------------+---+--------------------+
only showing top 10 rows



## `where`

- `DataFrame.where()`: **조건식을 만족**하는 **row**만 반환

In [12]:
# 나이(Age)가 40 초과인 선수만 추출
fifa_1.where(fifa_1.Age>40).show(10)

+-------------+---+--------------------+
|         Name|Age|                Club|
+-------------+---+--------------------+
|    J. Villar| 41|                NULL|
|     B. Nivet| 41|        ESTAC Troyes|
|     O. Pérez| 45|             Pachuca|
|     C. Muñoz| 41|CD Universidad de...|
|  S. Narazaki| 42|      Nagoya Grampus|
| H. Sulaimani| 41|           Ohod Club|
|     M. Tyler| 41| Peterborough United|
|    T. Warner| 44|  Accrington Stanley|
|K. Pilkington| 44|    Cambridge United|
+-------------+---+--------------------+



- column의 값이 str일 때 사용할 수 있는 조건식
> - `DataFrame['col'].isin(str1,str2,str3,...)`: 포함한 원소들(str1,str2,str3,..) 중 하나인 경우, `True`
> - `DataFrame['col'].startswith(str)`: str로 시작하는 문자열인 경우, `True`
> - `DataFrame['col'].endswith(str)`: str로 끝나는 문자열인 경우, `True`
> - `DataFrame['col'].like(str)`: str을 포함하는 문자열인 경우, `True`

In [13]:
# Club이 Juventus, Lazio, Real Madrid인 경우
fifa_1.where(fifa_1.Club.isin('Juventus', 'Lazio', 'Real Madrid')).show(10)

+-----------------+---+-----------+
|             Name|Age|       Club|
+-----------------+---+-----------+
|Cristiano Ronaldo| 33|   Juventus|
|        L. Modrić| 32|Real Madrid|
|     Sergio Ramos| 32|Real Madrid|
|         T. Kroos| 28|Real Madrid|
|        P. Dybala| 24|   Juventus|
|      T. Courtois| 26|Real Madrid|
|     G. Chiellini| 33|   Juventus|
|         Casemiro| 26|Real Madrid|
|             Isco| 26|Real Madrid|
|          Marcelo| 30|Real Madrid|
+-----------------+---+-----------+
only showing top 10 rows



In [14]:
# 성이 Lee 인 선수 행 추출
# -> Name이 'Lee'로 끝나는 선수
fifa_1.where(fifa_1.Name.endswith('Lee')).show(10)

+----------+---+-------------------+
|      Name|Age|               Club|
+----------+---+-------------------+
|    K. Lee| 30|Sheffield Wednesday|
|Kangin Lee| 17|        Valencia CF|
|    O. Lee| 26|Heart of Midlothian|
|    T. Lee| 32| Urawa Red Diamonds|
|    E. Lee| 23|         Luton Town|
+----------+---+-------------------+



In [15]:
# Club이름에 Madrid가 포함된 행 추출
# %의 위치에는 다른 문자 또는 문자열이 있을 수 있다는 의미
fifa_1.where(fifa_1.Club.like('%Madrid%')).show(10)

+------------+---+---------------+
|        Name|Age|           Club|
+------------+---+---------------+
|   L. Modrić| 32|    Real Madrid|
|Sergio Ramos| 32|    Real Madrid|
|    J. Oblak| 25|Atlético Madrid|
|    T. Kroos| 28|    Real Madrid|
|    D. Godín| 32|Atlético Madrid|
|A. Griezmann| 27|Atlético Madrid|
| T. Courtois| 26|    Real Madrid|
|    Casemiro| 26|    Real Madrid|
|        Isco| 26|    Real Madrid|
|     Marcelo| 30|    Real Madrid|
+------------+---+---------------+
only showing top 10 rows



In [16]:
fifa_1.where(fifa_1.Name.like('%Lee')).show(10)
# like('%Lee')는 endswith('Lee')와 동일함

+----------+---+-------------------+
|      Name|Age|               Club|
+----------+---+-------------------+
|    K. Lee| 30|Sheffield Wednesday|
|Kangin Lee| 17|        Valencia CF|
|    O. Lee| 26|Heart of Midlothian|
|    T. Lee| 32| Urawa Red Diamonds|
|    E. Lee| 23|         Luton Town|
+----------+---+-------------------+



#### **과제**
- `Age`가 38이상인 선수의 `Nationality, Name, Overall, Age`를 `Age`를 기준으로 **내림차순 정렬 후, 상위 10개 행**을 보이시오.

In [17]:
fifa.select("Nationality","Name","Overall","Age").where(fifa.Age>=38).orderBy(fifa['Age'].desc()).show(10)

+-----------------+-------------+-------+---+
|      Nationality|         Name|Overall|Age|
+-----------------+-------------+-------+---+
|           Mexico|     O. Pérez|     71| 45|
|Trinidad & Tobago|    T. Warner|     53| 44|
|          England|K. Pilkington|     48| 44|
|            Japan|  S. Narazaki|     65| 42|
|         Paraguay|    J. Villar|     77| 41|
|     Saudi Arabia| H. Sulaimani|     63| 41|
|           France|     B. Nivet|     71| 41|
|          England|     M. Tyler|     59| 41|
|        Argentina|     C. Muñoz|     68| 41|
|         Colombia|  B. Castillo|     66| 40|
+-----------------+-------------+-------+---+
only showing top 10 rows



## `Collect`

- cluster에 분산된 데이터를 모음
- `collect`를 이용하여 분산된 모든 데이터를 local로 가져오는 것은 위험할 수 있으므로(out of meory), 단독으로는 쓰이지 않음
- `collect`는 `Row` 객체를 원소로 하는 `list`로 반환함

In [18]:
fifa.select('Age','Name').collect()

[Row(Age=31, Name='L. Messi'),
 Row(Age=33, Name='Cristiano Ronaldo'),
 Row(Age=26, Name='Neymar Jr'),
 Row(Age=27, Name='De Gea'),
 Row(Age=27, Name='K. De Bruyne'),
 Row(Age=27, Name='E. Hazard'),
 Row(Age=32, Name='L. Modrić'),
 Row(Age=31, Name='L. Suárez'),
 Row(Age=32, Name='Sergio Ramos'),
 Row(Age=25, Name='J. Oblak'),
 Row(Age=29, Name='R. Lewandowski'),
 Row(Age=28, Name='T. Kroos'),
 Row(Age=32, Name='D. Godín'),
 Row(Age=32, Name='David Silva'),
 Row(Age=27, Name='N. Kanté'),
 Row(Age=24, Name='P. Dybala'),
 Row(Age=24, Name='H. Kane'),
 Row(Age=27, Name='A. Griezmann'),
 Row(Age=26, Name='M. ter Stegen'),
 Row(Age=26, Name='T. Courtois'),
 Row(Age=29, Name='Sergio Busquets'),
 Row(Age=31, Name='E. Cavani'),
 Row(Age=32, Name='M. Neuer'),
 Row(Age=30, Name='S. Agüero'),
 Row(Age=33, Name='G. Chiellini'),
 Row(Age=19, Name='K. Mbappé'),
 Row(Age=26, Name='M. Salah'),
 Row(Age=26, Name='Casemiro'),
 Row(Age=26, Name='J. Rodríguez'),
 Row(Age=27, Name='L. Insigne'),
 Row(Age=2

In [19]:
print(type(fifa.select('Age','Name').collect()))
print(type(fifa.select('Age','Name').collect()[0]))
print(fifa.select('Age','Name').collect()[0])

<class 'list'>
<class 'pyspark.sql.types.Row'>
Row(Age=31, Name='L. Messi')


In [20]:
# Row로부터 각 요소 추출
print(fifa.select('Age','Name').collect()[0]['Name'])
print(fifa.select('Age','Name').collect()[0][1])

L. Messi
L. Messi


#### **과제**
- `Age`가 20세 미만인 선수 중, `Potential`이 가장 큰 선수의 `Nationality`를 찾으시오.
- `DataFrame`에 `Potential, Age, Nationality`가 포함되도록 다시 추출하시오.

In [21]:
# Potential 기준 내림차순으로 정렬하면, Potential이 가장 큰 선수는 첫번째 행
# collect 후 인덱스 [0]으로 해당 행을 추출할 수 있음
fifa_n=fifa.select('Potential','Age','Nationality').where("Age<20").orderBy(fifa['Potential'].desc()).collect()
fifa_n[0]['Nationality']

'France'

In [22]:
# Potential 기준 오름차순 정렬 후, 마지막 행의 Nationality 추출
fifa.select('Potential','Age','Nationality').where(fifa.Age<20).orderBy(fifa.Potential).collect()[-1]['Nationality']

'France'